In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import numpy as np  # numbers
import pandas as pd  # tables

folder = "/kaggle/input/notebooks/raesiow/09-build-index/"  # the 09 output folder
embeddings = np.load(folder + "off_image_embeddings.npy")  # one row of 384 numbers per product picture
products = pd.read_csv(folder + "index_products.csv", dtype={"code": str})  # product list, barcodes kept as text

print("embeddings shape:", embeddings.shape)  # expect (71241, 384)
print("products shape:", products.shape)  # expect 71241 rows
print("same length:", len(products) == embeddings.shape[0])  # must be True

In [ ]:
!pip install -q faiss-cpu  # fast search library (Internet must be ON)

In [ ]:
import faiss  # fast nearest-neighbour search
import torch  # deep learning library
from transformers import AutoImageProcessor, AutoModel  # DINOv2 loader
from PIL import Image, ImageOps  # open images, apply phone rotation flag

index = faiss.IndexFlatIP(embeddings.shape[1])  # exact search by inner product
index.add(embeddings)  # put all vectors into the index
print("vectors in index:", index.ntotal)  # expect 71241

device = "cuda" if torch.cuda.is_available() else "cpu"  # use the GPU if there is one
print("device:", device)  # expect cuda
processor = AutoImageProcessor.from_pretrained("facebook/dinov2-small")  # resizes and normalises images
dino = AutoModel.from_pretrained("facebook/dinov2-small").to(device)  # the model itself
dino.eval()  # evaluation mode

def embed_image(img):  # turn one PIL image into one 384-number vector
    inputs = processor(images=img.convert("RGB"), return_tensors="pt").to(device)  # resize, normalise, to GPU
    with torch.no_grad():  # no gradients needed
        out = dino(**inputs)  # run DINOv2
    vec = torch.nn.functional.normalize(out.pooler_output, dim=1)  # image summary vector, length 1
    return vec.cpu().numpy().astype("float32")  # back to numpy

def search_image(img, k=5):  # the k catalogue products most similar by picture
    scores, rows = index.search(embed_image(img), k)  # top-k similarities and row numbers
    result = products.iloc[rows[0]].copy()  # look up those products
    result["similarity"] = scores[0]  # add the similarity column
    return result[["code", "product_name", "countries_en", "similarity"]]  # columns to see

In [ ]:
!pip install -q easyocr  # text reader (Internet must be ON)

In [ ]:
import re  # splitting text into words
from collections import Counter  # counting words
import easyocr  # text reader

reader = easyocr.Reader(["en"], gpu=True)  # English reader on the GPU

def words_of(text):  # the set of lowercase words of 3+ letters
    return set(re.findall(r"[a-z]{3,}", str(text).lower()))  # letters only

name_words = products["product_name"].apply(words_of)  # word set of every catalogue name
doc_count = Counter(w for ws in name_words for w in ws)  # how many names contain each word
idf = {w: np.log(len(products) / c) for w, c in doc_count.items()}  # rare words get bigger values

def read_pack_text(img, angles):  # read the front text at each allowed angle, keep the best angle
    best_angle, best_score, best_text = angles[0], -1, ""  # nothing picked yet
    for angle in angles:  # each angle we are allowed to try
        pieces = reader.readtext(np.array(img.rotate(angle, expand=True)))  # (box, text, confidence) pieces
        good = [(t, c) for box, t, c in pieces if c > 0.5]  # keep pieces the reader is fairly sure about
        score = sum(c for t, c in good)  # total confidence at this angle
        if score > best_score:  # better than what we have so far
            best_angle, best_score, best_text = angle, score, " ".join(t for t, c in good)  # remember it
    return best_angle, best_text  # the picked angle and its text

In [ ]:
def identify(img, upright=True, K=100, threshold=10):  # the TESTED rule: photo in, decision and top 3 out
    img = ImageOps.exif_transpose(img).convert("RGB")  # apply the phone's rotation flag
    small = img.copy()  # a copy for the text reader
    small.thumbnail((1600, 1600))  # shrink the copy to at most 1600 px
    angles = [0] if upright else [0, 90, 180, 270]  # upright photos: 0 only
    angle, text = read_pack_text(small, angles)  # pick the angle and read the pack text
    query = embed_image(img.rotate(angle, expand=True))  # embed the full-size photo
    sims, rows = index.search(query, K)  # shortlist of K look-alikes
    ocr_words = words_of(text)  # words read from the pack
    scored = []  # (word score, similarity, row) for every candidate
    for sim, row in zip(sims[0], rows[0]):  # each candidate
        matched = ocr_words & name_words[row]  # words shared with the pack text
        scored.append((sum(idf[w] for w in matched), float(sim), int(row)))  # score = sum of rarity values
    scored.sort(key=lambda s: (s[0], s[1]), reverse=True)  # best word score first, ties by similarity
    decision = "accept" if scored[0][0] >= threshold else "not sure"  # the fixed rule
    top3 = pd.DataFrame([{"row": r, "code": products.loc[r, "code"], "name": products.loc[r, "product_name"],
                          "word_score": round(s, 1), "similarity": round(sim, 3)} for s, sim, r in scored[:3]])
    return decision, angle, text, top3  # everything

In [ ]:
import os  # file listing

found_files = []  # all files in testset
for folder_, _, names in os.walk("/kaggle/input"):  # look through all inputs
    if "testset" in folder_:  # only the test set
        for name in names:  # each file
            found_files.append(os.path.join(folder_, name))  # remember the full path
key_path = [p for p in found_files if p.endswith(".csv")][0]  # the answer key CSV
key = pd.read_csv(key_path, dtype=str)  # read as text so barcodes stay intact
key.columns = key.columns.str.strip().str.lower()  # lowercase column names
photo_path = {os.path.basename(p).lower(): p for p in found_files if not p.endswith(".csv")}  # file name -> path

def norm(code):  # make a barcode comparable: text, no spaces, no leading zeros
    return str(code).strip().lstrip("0")  # clean it

print(len(key), "photos in the key;", len(photo_path), "photo files")  # expect 27 and 27

In [ ]:
rows_of_word = {}  # word -> list of catalogue rows whose name contains it
for row_number, ws in enumerate(name_words):  # every catalogue name
    for w in ws:  # every word in it
        rows_of_word.setdefault(w, []).append(row_number)  # remember the row
rows_of_word = {w: np.array(r) for w, r in rows_of_word.items()}  # lists to arrays for speed

def candidates(img, n=5):  # photo in, n candidate products out (whole-catalogue word search)
    img = ImageOps.exif_transpose(img).convert("RGB")  # apply the phone's rotation flag
    small = img.copy()  # a copy for the text reader
    small.thumbnail((1600, 1600))  # shrink it to at most 1600 px
    angle, text = read_pack_text(small, [0])  # upright photo: read at angle 0 only
    word_score = np.zeros(len(products))  # score of every catalogue product, starting at 0
    for w in words_of(text):  # each word read from the pack
        if w in rows_of_word:  # only words that appear in some catalogue name
            word_score[rows_of_word[w]] += idf[w]  # add the rarity value to every product with that word
    similarity = (embeddings @ embed_image(img)[0]).astype("float64")  # picture similarity to every product
    order = np.lexsort((similarity, word_score))[::-1][:n]  # best word score first, ties by picture
    table = products.iloc[order][["code", "product_name", "countries_en", "image_url"]].copy()  # candidate rows
    table["word_score"] = word_score[order].round(1)  # their word scores
    table["similarity"] = similarity[order].round(3)  # their picture similarities
    return table.reset_index(), text  # the table (original row number kept) and the text that was read

In [ ]:
import requests  # download catalogue pictures
from io import BytesIO  # treat bytes like a file
import matplotlib.pyplot as plt  # draw pictures

def show_candidates(file_name):  # draw one test photo next to its 5 candidates
    img = Image.open(photo_path[file_name.lower()])  # open the photo
    table, text = candidates(img)  # get the 5 candidates
    print("text read:", text[:150])  # what the reader saw
    fig, axes = plt.subplots(1, 6, figsize=(18, 3.6))  # one box for the photo, five for candidates
    axes[0].imshow(ImageOps.exif_transpose(img)); axes[0].set_title("YOUR PHOTO", fontsize=9)  # the photo
    for ax, (_, r) in zip(axes[1:], table.iterrows()):  # each candidate
        try:  # the picture might fail to download
            pic = Image.open(BytesIO(requests.get(r["image_url"], timeout=20).content))  # download it
            ax.imshow(pic)  # draw it
        except Exception:
            ax.text(0.5, 0.5, "no picture", ha="center")  # placeholder
        ax.set_title(f"{r['product_name'][:28]}\nscore {r['word_score']}", fontsize=8)  # name and score
    for ax in axes:
        ax.axis("off")  # hide the axes
    plt.show()  # display
    return table  # also return the table

_ = show_candidates(key["file"].iloc[0].strip())  # try the first test photo (plumbing check only, NOT a result)

In [ ]:
def picture_first(img, text, n=5, K=100):  # OLD design: shortlist K by picture, re-rank by shared words
    img = ImageOps.exif_transpose(img).convert("RGB")  # apply the phone's rotation flag
    sims, rows = index.search(embed_image(img), K)  # the K most picture-similar products
    ocr_words = words_of(text)  # words read from the pack
    scored = []  # (word score, similarity, row) for each shortlisted product
    for sim, row in zip(sims[0], rows[0]):  # each shortlisted product
        matched = ocr_words & name_words[row]  # words shared with the pack text
        scored.append((sum(idf[w] for w in matched), float(sim), int(row)))  # score = sum of rarity values
    scored.sort(key=lambda s: (s[0], s[1]), reverse=True)  # best word score first, ties by picture
    return [r for s, sim, r in scored[:n]]  # row numbers of the top n

def draw_candidates(axes, rows):  # draw 5 catalogue pictures with their names, one per box
    for ax, r in zip(axes, rows):  # each candidate
        try:  # the picture might fail to download
            ax.imshow(Image.open(BytesIO(requests.get(products.loc[r, "image_url"], timeout=20).content)))  # download and draw
        except Exception:
            ax.text(0.5, 0.5, "no picture", ha="center")  # placeholder
        ax.set_title(products.loc[r, "product_name"][:30], fontsize=8)  # product name only, no scores

compare_rows = {}  # file name -> (rows of old design, rows of new design), saved for judging
for number, file_name in enumerate(key["file"].str.strip()):  # each of the 27 photos
    img = Image.open(photo_path[file_name.lower()])  # open the photo
    small = ImageOps.exif_transpose(img).convert("RGB")  # rotation flag applied
    small.thumbnail((1600, 1600))  # smaller copy for the text reader
    _, text = read_pack_text(small, [0])  # read the pack text once, upright
    rows_a = picture_first(img, text)  # old design: picture shortlist, then words
    table_b, _ = candidates(img)  # new design: words over the whole catalogue
    rows_b = table_b["index"].tolist()  # its row numbers
    compare_rows[file_name] = (rows_a, rows_b)  # remember both
    fig, axes = plt.subplots(2, 6, figsize=(18, 7))  # two rows of six boxes
    axes[0, 0].imshow(ImageOps.exif_transpose(img))  # your photo, top left
    axes[0, 0].set_title(f"{number}: {file_name}", fontsize=9)  # photo number and file name
    axes[1, 0].text(0.5, 0.5, "ROW B\\nwords across\\nall names", ha="center", va="center")  # label for the bottom row
    draw_candidates(axes[0, 1:], rows_a)  # top row: picture first, then words
    draw_candidates(axes[1, 1:], rows_b)  # bottom row: words over the whole catalogue
    for ax in axes.ravel():
        ax.axis("off")  # hide all axes
    plt.suptitle("TOP ROW (A): picture first, then words   |   BOTTOM ROW (B): words over whole catalogue", fontsize=10)  # explain rows
    plt.show()  # display this photo

In [ ]:
# final verdicts per photo (number = photo number above each image): row A, row B
# E = exact, S = sibling, T = same type (different brand), N = none; number = position 1-5
verdicts = {
    0: ("E1", "E4"), 1: ("E1", "E4"), 2: ("N", "S1"), 3: ("E1", "N"), 4: ("E1", "E1"),  # photos 0-4
    5: ("T1", "T1"), 6: ("N", "N"), 7: ("S3", "E2"), 8: ("T1", "N"), 9: ("T3/T5", "E2"),  # photos 5-9
    10: ("T2", "N"), 11: ("E1/E2", "E1/E2"), 12: ("T1", "N"), 13: ("S1/S2/S3/S4", "S1"), 14: ("E2", "N"),  # photos 10-14
    15: ("S3", "S1"), 16: ("T2/T3/T4", "T4/T5"), 17: ("E1", "E1"), 18: ("T2", "T4"), 19: ("S3/S4", "E1"),  # photos 15-19
    20: ("N", "S1"), 21: ("T1/T2", "T1"), 22: ("E3", "N"), 23: ("N", "E2"), 24: ("S1", "E1"),  # photos 20-24
    25: ("T1", "N"), 26: ("E2", "N"),  # photos 25-26
}

table = pd.DataFrame([{"photo": i, "file": key["file"].iloc[i].strip(), "row_a": a, "row_b": b}  # one line per photo
                      for i, (a, b) in verdicts.items()])  # turn the dictionary into a table
table["a_letter"] = table["row_a"].str[0]  # first letter of the row A verdict
table["b_letter"] = table["row_b"].str[0]  # first letter of the row B verdict
table.to_csv("/kaggle/working/candidate_verdicts.csv", index=False)  # save the file

print(table["a_letter"].value_counts().reindex(list("ESTN")).fillna(0).astype(int).to_dict())  # row A counts, expect E9 S5 T9 N4
print(table["b_letter"].value_counts().reindex(list("ESTN")).fillna(0).astype(int).to_dict())  # row B counts, expect E10 S4 T4 N9
print("exact in either row:", int(((table["a_letter"] == "E") | (table["b_letter"] == "E")).sum()))  # expect 14
print("exact or sibling in either row:", int((table["a_letter"].isin(["E", "S"]) | table["b_letter"].isin(["E", "S"])).sum()))  # expect 18
print("same type or better in either row:", int(((table["a_letter"] != "N") | (table["b_letter"] != "N")).sum()))  # expect 26

In [ ]:
def final_candidates(img, n=5, n_from_a=3):  # the fixed rule: 3 from row A, then row B, no duplicates, 5 in total
    img = ImageOps.exif_transpose(img).convert("RGB")  # apply the phone's rotation flag
    small = img.copy()  # a copy for the text reader
    small.thumbnail((1600, 1600))  # shrink it to at most 1600 px
    _, text = read_pack_text(small, [0])  # read the pack text once, upright
    # row A: picture shortlist of 100, re-ranked by shared words (the 10 best kept as a reserve)
    list_a = picture_first(img, text, n=10)  # row numbers, best first
    # row B: words across the whole catalogue
    word_score = np.zeros(len(products))  # score of every catalogue product
    for w in words_of(text):  # each word read from the pack
        if w in rows_of_word:  # only words that appear in some name
            word_score[rows_of_word[w]] += idf[w]  # add the rarity value
    similarity = (embeddings @ embed_image(img)[0]).astype("float64")  # picture similarity to every product
    list_b = np.lexsort((similarity, word_score))[::-1][:10].tolist()  # row numbers, best first (10 kept)
    chosen = []  # the final list: (row number, where it came from)
    for r in list_a[:n_from_a]:  # first the top 3 of row A
        chosen.append((r, "A"))  # remember the source
    for r in list_b:  # then row B, best first
        if len(chosen) >= n:  # stop at 5
            break
        if r not in [c[0] for c in chosen]:  # skip duplicates
            chosen.append((r, "B"))  # remember the source
    for r, src in [(r, "A") for r in list_a] + [(r, "B") for r in list_b]:  # fill any gaps from the reserves
        if len(chosen) >= n:  # stop at 5
            break
        if r not in [c[0] for c in chosen]:  # skip duplicates
            chosen.append((r, src))  # remember the source
    return chosen, text  # list of (row, source) and the text that was read

def show_final(file_name):  # draw one photo with its final 5 candidates
    img = Image.open(photo_path[file_name.lower()])  # open the photo
    chosen, text = final_candidates(img)  # the 5 candidates
    print("text read:", text[:150])  # what the reader saw
    fig, axes = plt.subplots(1, 6, figsize=(18, 3.6))  # one box for the photo, five for candidates
    axes[0].imshow(ImageOps.exif_transpose(img))  # the photo
    axes[0].set_title(file_name[:30], fontsize=9)  # its name
    for ax, (r, src) in zip(axes[1:], chosen):  # each candidate
        try:  # the picture might fail to download
            ax.imshow(Image.open(BytesIO(requests.get(products.loc[r, "image_url"], timeout=20).content)))  # download and draw
        except Exception:
            ax.text(0.5, 0.5, "no picture", ha="center")  # placeholder
        ax.set_title(f"[{src}] " + products.loc[r, "product_name"][:26], fontsize=8)  # source and name
    for ax in axes:
        ax.axis("off")  # hide the axes
    plt.show()  # display

for i in [3, 7, 14]:  # plumbing check on three photos (NOT a result): one A-win, one B-win, one A-win
    show_final(key["file"].iloc[i].strip())  # draw it

In [ ]:
cat_path = "/kaggle/input/datasets/raesiow/catalogue-predictions-v2/catalogue_predictions_v2.csv"  # the catalogue with NOVA and GI predictions
cat = pd.read_csv(cat_path, dtype={"code": str})  # load it, barcodes as text
print(cat.shape)  # expect (71298, 17)
band_rank = {"low": 0, "medium": 1, "high": 2}  # lower is better
cat["band_rank"] = cat["gi_band_new"].map(band_rank)  # turn the band words into numbers
cat.loc[cat["gi_not_estimated"], "band_rank"] = 1  # hidden GI counts as medium, for ranking only
cat["gi_number_rank"] = cat["gi_number_new"].clip(0, 100)  # the number, clipped to 0-100 as shown in the app
cat.loc[cat["gi_not_estimated"], "gi_number_rank"] = 50  # hidden GI counts as 50, for ranking only

def recommend(code, n=3):  # a chosen product's barcode in, up to n better alternatives out
    own = cat[cat["code"] == str(code)]  # find the chosen product in the catalogue
    if len(own) == 0:  # barcode not found
        return None, "product not found in the catalogue"  # tell the caller
    own = own.iloc[0]  # take the single row
    category = own["pnns_groups_2"]  # its food category
    if pd.isna(category) or str(category).lower() == "unknown":  # no usable category
        return own, "no same-category alternative available"  # as agreed
    same = cat[(cat["pnns_groups_2"] == category) & (cat["code"] != own["code"])]  # same category, not itself
    better = same[(same["nova_pred"] < own["nova_pred"]) |  # strictly lower NOVA
                  ((same["nova_pred"] == own["nova_pred"]) & (same["band_rank"] < own["band_rank"]))]  # or same NOVA and lower GI band
    better = better.sort_values(["nova_pred", "band_rank", "gi_number_rank"])  # lowest NOVA, then band, then number
    if len(better) == 0:  # nothing better exists
        return own, "no better alternative found in this category"  # say so
    return own, better.head(n)  # the chosen product and its top alternatives

def show_reco(code):  # print a chosen product and its alternatives
    own, result = recommend(code)  # run the rule
    if own is None:  # not found
        print(result)  # message only
        return
    print("CHOSEN:", own["product_name"], "| category:", own["pnns_groups_2"], "| NOVA", own["nova_pred"],  # what was chosen
          "| GI band:", "not estimated" if own["gi_not_estimated"] else own["gi_band_new"])  # its GI band
    if isinstance(result, str):  # a message instead of a table
        print(" ->", result)  # print it
    else:  # a table of alternatives
        print(result[["product_name", "nova_pred", "gi_band_new", "gi_number_new", "gi_not_estimated"]].round(0).to_string(index=False))  # show them

# plumbing check on three products found by name (NOT a result)
for word in ["corn flakes", "white bread", "digestive"]:  # three food types
    hit = cat[cat["product_name"].str.contains(word, case=False, na=False) & (cat["pnns_groups_2"].notna())].head(1)  # first match
    if len(hit):  # found one
        show_reco(hit.iloc[0]["code"])  # show its alternatives
        print()  # blank line between
print(cat.columns.tolist())  # all column names
print(cat.head(3).T)  # the first 3 rows, turned sideways so each column is readable

In [ ]:
from sentence_transformers import SentenceTransformer  # the MiniLM text model used earlier in the project

mini = SentenceTransformer("all-MiniLM-L6-v2", device=device)  # load MiniLM on the GPU
name_vecs = mini.encode(cat["product_name"].fillna("").tolist(), batch_size=256,  # turn every product name into numbers
                        normalize_embeddings=True, show_progress_bar=False)  # length 1 so a dot product = similarity
print("name vectors:", name_vecs.shape)  # expect (71298, 384)

def recommend2(code, n=3, pool=50):  # chosen barcode in; alternatives out, from the pool most similar by name
    positions = np.where(cat["code"].values == str(code))[0]  # row position of the chosen product
    if len(positions) == 0:  # barcode not found
        return None, "product not found in the catalogue"  # tell the caller
    own_pos = positions[0]  # position of the chosen product
    own = cat.iloc[own_pos]  # its row
    category = own["pnns_groups_2"]  # its food category
    if pd.isna(category) or str(category).lower() == "unknown":  # no usable category
        return own, "no same-category alternative available"  # as agreed
    same_pos = np.where((cat["pnns_groups_2"].values == category) & (np.arange(len(cat)) != own_pos))[0]  # same category, not itself
    sims = name_vecs[same_pos] @ name_vecs[own_pos]  # name similarity to the chosen product
    keep = same_pos[np.argsort(-sims)[:pool]]  # the pool most similar by name
    similar = cat.iloc[keep].copy()  # their rows
    similar["name_similarity"] = np.sort(sims)[::-1][:pool].round(2)  # remember how similar each is
    better = similar[(similar["nova_pred"] < own["nova_pred"]) |  # strictly lower NOVA
                     ((similar["nova_pred"] == own["nova_pred"]) & (similar["band_rank"] < own["band_rank"]))]  # or same NOVA, lower GI band
    better = better.sort_values(["nova_pred", "band_rank", "gi_number_rank"])  # lowest NOVA, then band, then number
    if len(better) == 0:  # nothing better in the pool
        return own, "no better alternative among the most similar products"  # say so
    return own, better.head(n)  # the chosen product and its top alternatives

def show_reco2(code):  # print a chosen product and its alternatives
    own, result = recommend2(code)  # run the rule
    if own is None:  # not found
        print(result)  # message only
        return
    print("CHOSEN:", own["product_name"], "| category:", own["pnns_groups_2"], "| NOVA", own["nova_pred"],  # chosen product
          "| GI band:", "not estimated" if own["gi_not_estimated"] else own["gi_band_new"])  # its GI band
    if isinstance(result, str):  # a message instead of a table
        print(" ->", result)  # print it
    else:  # a table of alternatives
        print(result[["product_name", "name_similarity", "nova_pred", "gi_band_new", "gi_number_new"]].round(0).to_string(index=False))  # show them

for word in ["corn flakes", "white bread", "digestive"]:  # same three food types as before
    hit = cat[cat["product_name"].str.contains(word, case=False, na=False) & cat["pnns_groups_2"].notna()].head(1)  # first match
    if len(hit):  # found one
        show_reco2(hit.iloc[0]["code"])  # show its alternatives
        print()  # blank line between

In [ ]:
def recommend3(code, n=3, pool=200):  # v3: chosen barcode in; alternatives out
    positions = np.where(cat["code"].values == str(code))[0]  # row position of the chosen product
    own_pos = positions[0]  # position of the chosen product
    own = cat.iloc[own_pos]  # its row
    category = own["pnns_groups_2"]  # its food category
    if pd.isna(category) or str(category).lower() == "unknown":  # no usable category
        return own, "no same-category alternative available"  # as agreed
    same_pos = np.where((cat["pnns_groups_2"].values == category) & (np.arange(len(cat)) != own_pos))[0]  # same category, not itself
    sims = name_vecs[same_pos] @ name_vecs[own_pos]  # name similarity to the chosen product
    keep = same_pos[np.argsort(-sims)[:pool]]  # the pool most similar by name
    similar = cat.iloc[keep].copy()  # their rows
    similar["name_similarity"] = np.sort(sims)[::-1][:pool].round(2)  # remember the similarity
    lower_nova = similar["nova_pred"] < own["nova_pred"]  # strictly lower NOVA
    if own["gi_not_estimated"]:  # the chosen product's GI is hidden
        better = similar[lower_nova]  # only NOVA can count
    else:  # GI is shown
        lower_gi = ((similar["nova_pred"] == own["nova_pred"]) & (~similar["gi_not_estimated"]) &  # same NOVA, GI shown
                    (similar["gi_number_rank"] <= own["gi_number_rank"] - 10))  # and GI at least 10 lower
        better = similar[lower_nova | lower_gi]  # either way counts
    better = better.sort_values(["nova_pred", "gi_number_rank"])  # lowest NOVA, then lowest GI number
    if len(better) == 0:  # nothing better in the pool
        return own, "no better alternative found"  # say so
    return own, better.head(n)  # the chosen product and its top alternatives

def show_reco3(code):  # print a chosen product and its alternatives
    own, result = recommend3(code)  # run the rule
    print("CHOSEN:", own["product_name"], "| category:", own["pnns_groups_2"], "| NOVA", own["nova_pred"],  # chosen product
          "| GI:", "not estimated" if own["gi_not_estimated"] else round(own["gi_number_new"]))  # its GI number
    if isinstance(result, str):  # a message instead of a table
        print(" ->", result)  # print it
    else:  # a table of alternatives
        shown = result[["product_name", "name_similarity", "nova_pred", "gi_number_new"]].copy()  # columns to show
        shown["gi_number_new"] = shown["gi_number_new"].round(0)  # GI to whole numbers
        print(shown.to_string(index=False))  # print the table
    print()  # blank line between products

test_words = ["yoghurt", "white bread", "corn flakes", "digestive", "peanut butter",  # the 10 fixed words
              "muesli", "pasta", "white rice", "cheese slices", "tomato sauce"]  # do not change after seeing results
for word in test_words:  # each word
    matches = cat[cat["product_name"].str.contains(word, case=False, na=False) & cat["pnns_groups_2"].notna()]  # known category only
    if len(matches):  # found some
        show_reco3(matches.sample(1, random_state=42).iloc[0]["code"])  # one random product, seed fixed

In [ ]:
import os  # file listing
import requests  # download pictures
from io import BytesIO  # treat bytes like a file
from sentence_transformers import SentenceTransformer  # the MiniLM text model

# --- the catalogue with NOVA and GI predictions (found automatically) ---
cat_path = None  # not found yet
for folder_, _, names in os.walk("/kaggle/input"):  # look through every input folder
    for name in names:  # each file
        if name == "catalogue_predictions_v2.csv":  # the file we want
            cat_path = os.path.join(folder_, name)  # remember the path
print("catalogue file:", cat_path)  # show it
cat = pd.read_csv(cat_path, dtype={"code": str})  # load it, barcodes as text
band_rank = {"low": 0, "medium": 1, "high": 2}  # lower is better
cat["band_rank"] = cat["gi_band_new"].map(band_rank)  # band words to numbers
cat.loc[cat["gi_not_estimated"], "band_rank"] = 1  # hidden GI counts as medium, for ranking only
cat["gi_number_rank"] = cat["gi_number_new"].clip(0, 100)  # GI number clipped to 0-100
cat.loc[cat["gi_not_estimated"], "gi_number_rank"] = 50  # hidden GI counts as 50, for ranking only

# --- name vectors for the recommendation and the name search ---
mini = SentenceTransformer("all-MiniLM-L6-v2", device=device)  # MiniLM on the GPU
name_vecs = mini.encode(cat["product_name"].fillna("").tolist(), batch_size=256,  # every product name as numbers
                        normalize_embeddings=True, show_progress_bar=False)  # length 1
print("name vectors:", name_vecs.shape)  # expect (71298, 384)

# --- row A of the candidate list: picture shortlist, then words ---
def picture_first(img, text, n=5, K=100):  # old design: shortlist K by picture, re-rank by shared words
    img = ImageOps.exif_transpose(img).convert("RGB")  # apply the phone's rotation flag
    sims, rows = index.search(embed_image(img), K)  # the K most picture-similar products
    ocr_words = words_of(text)  # words read from the pack
    scored = []  # (word score, similarity, row) for each shortlisted product
    for sim, row in zip(sims[0], rows[0]):  # each shortlisted product
        matched = ocr_words & name_words[row]  # words shared with the pack text
        scored.append((sum(idf[w] for w in matched), float(sim), int(row)))  # score = sum of rarity values
    scored.sort(key=lambda s: (s[0], s[1]), reverse=True)  # best word score first, ties by picture
    return [r for s, sim, r in scored[:n]]  # row numbers of the top n

# --- the fixed 3+2 candidate rule ---
def final_candidates(img, n=5, n_from_a=3):  # 3 from row A, then row B, no duplicates, 5 in total
    img = ImageOps.exif_transpose(img).convert("RGB")  # apply the phone's rotation flag
    small = img.copy()  # a copy for the text reader
    small.thumbnail((1600, 1600))  # shrink it to at most 1600 px
    _, text = read_pack_text(small, [0])  # read the pack text once, upright
    list_a = picture_first(img, text, n=10)  # row A, best first (10 kept as reserve)
    word_score = np.zeros(len(products))  # score of every catalogue product
    for w in words_of(text):  # each word read from the pack
        if w in rows_of_word:  # only words that appear in some name
            word_score[rows_of_word[w]] += idf[w]  # add the rarity value
    similarity = (embeddings @ embed_image(img)[0]).astype("float64")  # picture similarity to every product
    list_b = np.lexsort((similarity, word_score))[::-1][:10].tolist()  # row B, best first (10 kept)
    chosen = [(r, "A") for r in list_a[:n_from_a]]  # first the top 3 of row A
    for r in list_b:  # then row B, best first
        if len(chosen) >= n:  # stop at 5
            break
        if r not in [c[0] for c in chosen]:  # skip duplicates
            chosen.append((r, "B"))  # remember the source
    for r, src in [(r, "A") for r in list_a] + [(r, "B") for r in list_b]:  # fill any gaps from the reserves
        if len(chosen) >= n:  # stop at 5
            break
        if r not in [c[0] for c in chosen]:  # skip duplicates
            chosen.append((r, src))  # remember the source
    return chosen, text  # list of (row, source) and the text read

# --- the recommendation rule v3 (fixed rule) ---
def recommend3(code, n=3, pool=200):  # chosen barcode in; alternatives out
    positions = np.where(cat["code"].values == str(code))[0]  # row position of the chosen product
    own_pos = positions[0]  # position of the chosen product
    own = cat.iloc[own_pos]  # its row
    category = own["pnns_groups_2"]  # its food category
    if pd.isna(category) or str(category).lower() == "unknown":  # no usable category
        return own, "no same-category alternative available"  # as agreed
    same_pos = np.where((cat["pnns_groups_2"].values == category) & (np.arange(len(cat)) != own_pos))[0]  # same category, not itself
    sims = name_vecs[same_pos] @ name_vecs[own_pos]  # name similarity to the chosen product
    keep = same_pos[np.argsort(-sims)[:pool]]  # the pool most similar by name
    similar = cat.iloc[keep].copy()  # their rows
    similar["name_similarity"] = np.sort(sims)[::-1][:pool].round(2)  # remember the similarity
    lower_nova = similar["nova_pred"] < own["nova_pred"]  # strictly lower NOVA
    if own["gi_not_estimated"]:  # the chosen product's GI is hidden
        better = similar[lower_nova]  # only NOVA can count
    else:  # GI is shown
        lower_gi = ((similar["nova_pred"] == own["nova_pred"]) & (~similar["gi_not_estimated"]) &  # same NOVA, GI shown
                    (similar["gi_number_rank"] <= own["gi_number_rank"] - 10))  # and GI at least 10 lower
        better = similar[lower_nova | lower_gi]  # either way counts
    better = better.sort_values(["nova_pred", "gi_number_rank"])  # lowest NOVA, then lowest GI number
    if len(better) == 0:  # nothing better in the pool
        return own, "no better alternative found"  # say so
    return own, better.head(n)  # the chosen product and its top alternatives
print("ready")  # done

In [ ]:
import gradio as gr  # the app library

code_to_pos = {c: i for i, c in enumerate(cat["code"])}  # barcode -> row position in the catalogue table

def picture_of(code):  # the catalogue picture of a product, or a white square if it fails
    url = cat.iloc[code_to_pos[code]]["image_url"]  # picture web address
    try:  # the download might fail
        return Image.open(BytesIO(requests.get(url, timeout=20).content)).convert("RGB")  # download and open it
    except Exception:
        return Image.new("RGB", (200, 200), "white")  # placeholder

def to_gallery(codes):  # list of (picture, caption) for the gallery
    return [(picture_of(c), cat.iloc[code_to_pos[c]]["product_name"][:40]) for c in codes]  # one per product

def find_candidates(photo):  # photo in -> 5 candidates out
    if photo is None:  # nothing uploaded
        return [], [], "No photo yet."  # tell the user
    chosen, text = final_candidates(photo)  # the fixed 3+2 rule
    codes = [products.loc[r, "code"] for r, src in chosen]  # barcodes of the candidates
    codes = [c for c in codes if c in code_to_pos]  # keep those that are in the catalogue table
    return to_gallery(codes), codes, "Text read from the pack: " + text[:120]  # gallery, codes, text

def search_by_name(query):  # typed name in -> 5 most similar product names out
    vec = mini.encode([query], normalize_embeddings=True)[0]  # turn the typed text into numbers
    top = np.argsort(-(name_vecs @ vec))[:5]  # the 5 most similar names
    codes = cat["code"].iloc[top].tolist()  # their barcodes
    return to_gallery(codes), codes, "Search by name: " + query  # gallery, codes, text

def describe(codes, evt: gr.SelectData):  # the user tapped a candidate -> show NOVA, GI and alternatives
    own, result = recommend3(codes[evt.index])  # the chosen product and its alternatives
    lines = [f"### {own['product_name']}",  # product name
             f"**Processing level (NOVA):** group {int(own['nova_pred'])} (model confidence {own['nova_confidence']:.0%})"]  # NOVA
    if own["gi_not_estimated"]:  # GI is hidden for low-carb foods and sauces
        lines.append("**Glycaemic index:** not estimated for this kind of food (low carbohydrate, or a sauce).")  # explain
    else:  # GI is shown
        shown = min(max(own["gi_number_new"], 0), 100)  # clip to 0-100
        lines.append(f"**Glycaemic index band:** {own['gi_band_new']} (approximately {shown:.0f})")  # band and number
    lines.append("\n**Possible alternatives (based on predicted NOVA and GI):**")  # heading
    if isinstance(result, str):  # a message instead of a table
        lines.append("- " + result)  # print it
    else:  # a table of alternatives
        for _, r in result.iterrows():  # each alternative
            gi_text = "not estimated" if r["gi_not_estimated"] else f"approx. {min(max(r['gi_number_new'], 0), 100):.0f}"  # its GI text
            lines.append(f"- {r['product_name']} (NOVA {int(r['nova_pred'])}, GI {gi_text})")  # one line each
    lines.append("\n*Model estimates for guidance only, not medical advice.*")  # the caveat
    return "\n".join(lines)  # one text block

In [ ]:
def gi_text(row):  # the GI text for one product: band always, number only if it agrees with the band
    if row["gi_not_estimated"]:  # hidden for low-carb foods and sauces
        return "not estimated"  # say so
    number = min(max(row["gi_number_new"], 0), 100)  # clip to 0-100
    if row["gi_band_new"] == row["gi_band_from_number"]:  # the two models agree
        return f"{row['gi_band_new']} (approx. {number:.0f})"  # band and number
    return f"{row['gi_band_new']}"  # band only, no contradicting number

def describe(codes, evt: gr.SelectData):  # the user tapped a candidate -> show NOVA, GI and alternatives
    own, result = recommend3(codes[evt.index])  # the chosen product and its alternatives
    lines = [f"### {own['product_name']}",  # product name
             f"**Processing level (NOVA):** group {int(own['nova_pred'])} (model confidence {own['nova_confidence']:.0%})",  # NOVA
             f"**Glycaemic index:** {gi_text(own)}"]  # GI band, and number if consistent
    lines.append("\n**Possible alternatives (based on predicted NOVA and GI):**")  # heading
    if isinstance(result, str):  # a message instead of a table
        lines.append("- " + result)  # print it
    else:  # a table of alternatives
        for _, r in result.iterrows():  # each alternative
            lines.append(f"- {r['product_name']} (NOVA {int(r['nova_pred'])}, GI {gi_text(r)})")  # one line each
    lines.append("\n*Model estimates for guidance only, not medical advice.*")  # the caveat
    return "\n".join(lines)  # one text block
print("display fix added")  # done

In [ ]:
try:  # close the running app first
    demo.close()  # stops the old app
except Exception:
    pass  # nothing to close, fine

with gr.Blocks() as demo:  # the app layout
    gr.Markdown("# Food Scanner\nPhoto of the front of the pack, held upright.")  # title
    photo = gr.Image(type="pil", label="Photo of the product")  # upload box
    find_btn = gr.Button("Find product")  # start button
    read_text = gr.Markdown()  # shows the text read from the pack
    gallery = gr.Gallery(label="Is your product here? Tap it.", columns=5, height=260, allow_preview=False)  # the 5 candidates
    codes_state = gr.State([])  # remembers the candidates' barcodes
    query = gr.Textbox(label="Not here? Type the product name")  # fallback box
    search_btn = gr.Button("Search by name")  # fallback button
    result_md = gr.Markdown()  # NOVA, GI and alternatives appear here
    find_btn.click(find_candidates, photo, [gallery, codes_state, read_text])  # photo -> candidates
    search_btn.click(search_by_name, query, [gallery, codes_state, read_text])  # typed name -> candidates
    gallery.select(describe, codes_state, result_md)  # tap -> results

demo.launch(share=True)  # start the app again and print a new link

In [ ]:
def gi_text(row):  # the GI text for one product: band and number if they agree, otherwise say so
    if row["gi_not_estimated"]:  # hidden for low-carb foods and sauces
        return "not estimated"  # say so
    number = min(max(row["gi_number_new"], 0), 100)  # clip to 0-100
    if row["gi_band_new"] == row["gi_band_from_number"]:  # the two models agree
        return f"{row['gi_band_new']} (approx. {number:.0f})"  # band and number
    return "uncertain (the two estimates disagree)"  # honest message when they disagree
print("gi_text updated")  # done